# Alpamayo-v1.5 Gradient-Guided BFA Demo

Targets the **expert denoiser** and **action projections** in BF16.
Ranks bit-flip coordinates by a first-order loss estimate (`grad · Δw`),
then for each (bit, coord) measures the actual post-flip loss delta.

Output: `bfa_results.json` with per-trial `{bit, flat_idx, delta_loss, ...}`.
After the BFA sweep, selected high-impact flips are replayed through full
trajectory inference and saved to `bfa_trajectory_results.pt` for plotting.

In [ ]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.load_physical_aiavdataset import load_physical_aiavdataset
from alpamayo1_5 import helper

import bfa_utils as bfa        # sits next to this notebook

In [ ]:
# Pin to a specific GPU. Edit GPU_ID per session — never rely on cuda:0
# defaults on the shared 8×H20 box. All bfa.* calls below take DEVICE
# explicitly so we never silently land on someone else's GPU.
GPU_ID = int(os.environ.get("ALPAMAYO_GPU", "0"))
DEVICE = torch.device(f"cuda:{GPU_ID}")
print(f"Using device: {DEVICE} (visible CUDA devices: {torch.cuda.device_count()})")

In [ ]:
model = Alpamayo1_5.from_pretrained("nvidia/Alpamayo-1.5-10B", dtype=torch.bfloat16).to(DEVICE)
model.eval()
# We need requires_grad on target weights for bit-grad.
# (HF models default to requires_grad=True, so this is usually a no-op.)
for p in model.parameters():
    p.requires_grad_(True)

processor = helper.get_processor(model.tokenizer)

clip_ids = pd.read_parquet("clip_ids.parquet")["clip_id"].tolist()
clip_id = clip_ids[774]   # same default as inference.ipynb
data = load_physical_aiavdataset(clip_id)

messages = helper.create_message(
    data["image_frames"].flatten(0, 1),
    camera_indices=data["camera_indices"],
)
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors="pt",
)
model_inputs = helper.to_device(
    {
        "tokenized_data": inputs,
        "ego_history_xyz": data["ego_history_xyz"],
        "ego_history_rot": data["ego_history_rot"],
    },
    DEVICE,
)

In [ ]:
# GT action in the same form the diffusion model is trained on.
gt_action = model.action_space.traj_to_action(
    traj_history_xyz=data["ego_history_xyz"].to(DEVICE).to(torch.float32),
    traj_history_rot=data["ego_history_rot"].to(DEVICE).to(torch.float32),
    traj_future_xyz=data["ego_future_xyz"].to(DEVICE).to(torch.float32),
    traj_future_rot=data["ego_future_rot"].to(DEVICE).to(torch.float32),
)
# action_space returns (B, 1, n_waypoints, 2); squeeze traj_group dim → (B, n_waypoints, 2)
gt_action = gt_action.view(-1, *model.action_space.get_action_space_dims())
print("gt_action:", gt_action.shape, gt_action.dtype)

with torch.autocast(DEVICE.type, dtype=torch.bfloat16):
    ctx = bfa.build_fm_context(model, model_inputs, gt_action)

# Fix a noise sample so every trial uses the SAME x_t for comparable losses.
with torch.cuda.device(DEVICE):
    torch.cuda.manual_seed(42)
fixed_noise = torch.randn_like(ctx.gt_action)
T_VAL = 0.5

def loss_fn():
    with torch.autocast(DEVICE.type, dtype=torch.bfloat16):
        return bfa.fm_one_step_loss(model, ctx, t_val=T_VAL, noise=fixed_noise)

baseline = loss_fn().item()
print("baseline FM loss:", baseline)

In [ ]:
targets = bfa.collect_target_linears(model)
print(f"{len(targets)} target Linear modules")
print("first 5:", list(targets.keys())[:5])

# Single backward pass — grads are cloned out, so we can modify weights later.
t0 = time.time()
grads = bfa.compute_clean_grads(model, targets, loss_fn)
print(f"backward done in {time.time()-t0:.1f}s; "
      f"example grad norm: {list(grads.values())[0].norm().item():.4f}")

## Smoke tests

Lightweight checks. Run these BEFORE the main BFA loop — if any fails,
investigate before spending ~1 hour on the full sweep.

In [ ]:
# Cell 10: bit-flip primitives work as expected
w = torch.tensor([1.0], dtype=torch.bfloat16, device=DEVICE)

# sign flip: 1.0 → -1.0
assert bfa.bf16_xor_bit_all(w, 15).item() == -1.0

# exponent MSB flip: 1.0 has biased exp 0x7F = 01111111
#   flipping bit 14 → 11111111 → +Inf or NaN (non-finite)
flipped = bfa.bf16_xor_bit_all(w, 14)
assert not torch.isfinite(flipped).item()

# mantissa LSB flip: 1.0 → 1 + 2^-7
flipped = bfa.bf16_xor_bit_all(w, 0)
assert abs(flipped.float().item() - (1.0 + 2**-7)) < 1e-6

print("bit-flip correctness: OK")

In [ ]:
# Cell 11: flip-then-restore returns the weight tensor to its exact pre-flip state
mod = list(targets.values())[0]
snapshot = mod.weight.data.clone()

orig, flipped = bfa.bf16_flip_one(mod.weight.data, flat_idx=123, bit=14)
assert not torch.equal(mod.weight.data, snapshot), (
    "flip produced no change — is flat_idx=123 valid for this module?"
)
bfa.restore_one(mod.weight.data, 123, orig)
assert torch.equal(mod.weight.data, snapshot), "restore did not reproduce clean weight"

print("restore: OK")

In [ ]:
# Cell 12: baseline loss is deterministic across calls with fixed noise,
# and gradient-guided top-k is at least as strong as a random coord at the
# same bit. (Not a strict assertion — single-sample; we expect top-k to win
# on average in the full loop.)
with torch.no_grad():
    l1 = loss_fn().item()
    l2 = loss_fn().item()
assert abs(l1 - l2) < 1e-5, (l1, l2)
print(f"baseline determinism: OK ({l1})")

bit = 14   # exponent MSB — catastrophic bit
name = list(targets.keys())[0]
mod = targets[name]
flat_idx, _ = bfa.topk_bitflip_coords(mod.weight.data, grads[name], bit=bit, k=1)
r_top = bfa.measure_flipped_loss(model, ctx, mod, flat_idx[0].item(), bit, loss_fn)

rng = np.random.default_rng(0)
random_idx = int(rng.integers(0, mod.weight.numel()))
r_rand = bfa.measure_flipped_loss(model, ctx, mod, random_idx, bit, loss_fn)

print(f"top-k delta : {r_top['post_loss'] - baseline:.4g}")
print(f"random delta: {r_rand['post_loss'] - baseline:.4g}")

In [ ]:
BITS = list(range(16))      # full BF16 sweep. Start with [14, 15] for a quick smoke.
TOP_K_PER_MODULE = 20       # per (module, bit). Scale up after smoke test.
MODULE_SUBSET = list(targets.keys())   # or filter to a few names for speed

results = []
t_start = time.time()
for bit in BITS:
    for name in MODULE_SUBSET:
        mod = targets[name]
        w = mod.weight.data
        g = grads[name]
        flat_idx, bg_vals = bfa.topk_bitflip_coords(w, g, bit=bit, k=TOP_K_PER_MODULE)
        for i, idx in enumerate(flat_idx.tolist()):
            r = bfa.measure_flipped_loss(
                model, ctx, mod, flat_idx=idx, bit=bit, loss_fn=loss_fn,
            )
            r.update({
                "module": name,
                "rank": i,
                "predicted_bit_grad": float(bg_vals[i].item()),
                "baseline_loss": baseline,
                "delta_loss": r["post_loss"] - baseline,
            })
            results.append(r)
    print(f"bit {bit} done, {len(results)} trials, {time.time()-t_start:.0f}s elapsed")

out_path = Path("bfa_results.json")
out_path.write_text(json.dumps(results, indent=2))
print(f"saved {len(results)} trials to {out_path}")

In [ ]:
df = pd.DataFrame(results)
print(df.groupby("bit")["delta_loss"].describe()[["count","mean","max"]])

# Top-10 worst flips overall (finite post_loss only — bit-14 exponent flips often go inf)
df_finite = df[df["post_loss_finite"] == 1.0]
print("\nTop-10 single-bit flips by delta_loss (finite-loss only):")
print(df_finite.sort_values("delta_loss", ascending=False).head(10)[
    ["module","bit","flat_idx","orig_value","flipped_value","delta_loss"]
])
n_inf = int((df["post_loss_finite"] == 0.0).sum())
print(f"\n({n_inf} additional trials produced non-finite loss — see post_loss_finite column)")


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
df.boxplot(column="delta_loss", by="bit", ax=ax)
ax.set_yscale("symlog")
ax.set_title("Per-bit delta_loss distribution (BF16, gradient-guided top-k)")
ax.set_xlabel("bit index (0=mantissa LSB, 14=exp MSB, 15=sign)")
ax.set_ylabel("post_loss - baseline_loss")
plt.suptitle("")
plt.tight_layout()
plt.savefig("bfa_per_bit.png", dpi=120)
plt.show()

## Full trajectory rollouts for selected BFA flips

The FM-loss sweep above only measures the surrogate loss. This post-pass replays
the clean model and the highest-delta finite BFA flips through the standard
`sample_trajectories_from_data_with_vlm_rollout` path used by
`inference_cam_num.ipynb`, then saves the predicted trajectories needed for
the plot below.

In [ ]:
TRAJ_TOP_N = 5
TRAJ_NUM_SAMPLES = 16
TRAJ_SEED = 42
# Bit 14 is the BF16 exponent MSB. It often produces inf/NaN or saturated
# full-rollout failures, so keep it out of the trajectory overlay by default.
# Set this to set() if you specifically want to visualize catastrophic flips.
TRAJ_EXCLUDE_BITS = {14}
TRAJ_REQUIRE_FINITE_FLIPPED_VALUE = True
TRAJ_OUT_PATH = Path("bfa_trajectory_results.pt")
TRAJ_METRICS_PATH = Path("bfa_trajectory_metrics.json")


def set_rollout_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        with torch.cuda.device(DEVICE):
            torch.cuda.manual_seed(seed)


def run_full_rollout(seed=TRAJ_SEED):
    set_rollout_seed(seed)
    with torch.no_grad(), torch.autocast(DEVICE.type, dtype=torch.bfloat16):
        pred_xyz, pred_rot, extra = model.sample_trajectories_from_data_with_vlm_rollout(
            data=model_inputs,
            top_p=0.98,
            temperature=0.6,
            num_traj_samples=TRAJ_NUM_SAMPLES,
            max_generation_length=256,
            return_extra=True,
        )
    return pred_xyz.float().cpu(), pred_rot.float().cpu(), extra


def trajectory_metrics(pred_xyz):
    gt_xy = data["ego_future_xyz"].cpu()[0, 0, :, :2].numpy()
    pred_xy = pred_xyz[0, 0, :, :, :2].numpy()
    valid = np.isfinite(pred_xy).all(axis=(1, 2))
    if not valid.any():
        return {
            "valid_samples": 0,
            "minADE_m": None,
            "meanADE_m": None,
            "minFDE_m": None,
            "meanFDE_m": None,
        }

    pred_valid = pred_xy[valid]
    ade = np.linalg.norm(pred_valid - gt_xy[None, ...], axis=-1).mean(axis=-1)
    fde = np.linalg.norm(pred_valid[:, -1, :] - gt_xy[-1, :][None, :], axis=-1)
    return {
        "valid_samples": int(valid.sum()),
        "minADE_m": float(ade.min()),
        "meanADE_m": float(ade.mean()),
        "minFDE_m": float(fde.min()),
        "meanFDE_m": float(fde.mean()),
    }


def trajectory_delta_metrics(pred_xyz, ref_xyz):
    pred_xy = pred_xyz[0, 0, :, :, :2].numpy()
    ref_xy = ref_xyz[0, 0, :, :, :2].numpy()
    valid = np.isfinite(pred_xy).all(axis=(1, 2)) & np.isfinite(ref_xy).all(axis=(1, 2))
    if not valid.any():
        return {
            "valid_samples": 0,
            "max_abs_xy_delta_m": None,
            "mean_l2_xy_delta_m": None,
            "max_endpoint_l2_delta_m": None,
        }

    delta = pred_xy[valid] - ref_xy[valid]
    l2 = np.linalg.norm(delta, axis=-1)
    endpoint_l2 = np.linalg.norm(delta[:, -1, :], axis=-1)
    return {
        "valid_samples": int(valid.sum()),
        "max_abs_xy_delta_m": float(np.abs(delta).max()),
        "mean_l2_xy_delta_m": float(l2.mean()),
        "max_endpoint_l2_delta_m": float(endpoint_l2.max()),
    }


def jsonable_record(record):
    out = {}
    for key, value in record.items():
        if pd.isna(value):
            out[key] = None
        elif isinstance(value, np.integer):
            out[key] = int(value)
        elif isinstance(value, np.floating):
            out[key] = float(value)
        else:
            out[key] = value
    return out


def select_flips_for_trajectory_rollout(df_finite, top_n=TRAJ_TOP_N):
    candidates = df_finite.copy()
    excluded = pd.DataFrame(columns=candidates.columns)

    if TRAJ_REQUIRE_FINITE_FLIPPED_VALUE and "flipped_value" in candidates:
        finite_weight = np.isfinite(candidates["flipped_value"].to_numpy(dtype=float))
        excluded = pd.concat([excluded, candidates[~finite_weight]])
        candidates = candidates[finite_weight]

    if TRAJ_EXCLUDE_BITS:
        bit_allowed = ~candidates["bit"].isin(TRAJ_EXCLUDE_BITS)
        excluded = pd.concat([excluded, candidates[~bit_allowed]])
        candidates = candidates[bit_allowed]

    candidates = candidates.sort_values("delta_loss", ascending=False)
    # One representative per bit first, so the plot does not become five
    # copies of the same catastrophic exponent-bit failure mode.
    selected = candidates.drop_duplicates(subset=["bit"]).head(top_n)
    if len(selected) < top_n:
        fill = candidates.drop(index=selected.index, errors="ignore").head(top_n - len(selected))
        selected = pd.concat([selected, fill])

    if selected.empty:
        # Fallback keeps the cell usable even if every candidate was excluded.
        selected = df_finite.sort_values("delta_loss", ascending=False).head(top_n)
    return selected, excluded.sort_values("delta_loss", ascending=False)


selected_df, excluded_rollout_df = select_flips_for_trajectory_rollout(df_finite)
selected_flips = [
    jsonable_record(record)
    for record in selected_df.to_dict(orient="records")
]
print("Selected flips for full trajectory rollout:")
if selected_flips:
    print(selected_df[["module", "bit", "flat_idx", "flipped_value", "delta_loss"]])
    if not excluded_rollout_df.empty:
        print("\nTop excluded catastrophic/non-finite candidates:")
        print(excluded_rollout_df[["module", "bit", "flat_idx", "flipped_value", "delta_loss"]].head(10))
else:
    print("No finite-loss BFA flips found; only clean rollout will be saved.")

trajectory_results = {}
clean_pred_xyz, clean_pred_rot, clean_extra = run_full_rollout()
trajectory_results["clean"] = {
    "label": "clean",
    "trial": None,
    "pred_xyz": clean_pred_xyz,
    "pred_rot": clean_pred_rot,
    "cot": clean_extra["cot"][0].tolist() if "cot" in clean_extra else None,
    "metrics": trajectory_metrics(clean_pred_xyz),
    "delta_vs_clean": trajectory_delta_metrics(clean_pred_xyz, clean_pred_xyz),
}

for trial_idx, trial in enumerate(selected_flips, start=1):
    name = trial["module"]
    bit = int(trial["bit"])
    flat_idx = int(trial["flat_idx"])
    label = f"bfa_{trial_idx}: bit{bit} {name}[{flat_idx}]"
    mod = targets[name]

    orig, flipped = bfa.bf16_flip_one(mod.weight.data, flat_idx=flat_idx, bit=bit)
    try:
        current = mod.weight.data.view(-1)[flat_idx].detach().clone()
        flip_active = bool(torch.equal(current, flipped))
        with torch.no_grad():
            fm_loss_check = loss_fn().item()
        pred_xyz, pred_rot, extra = run_full_rollout()
        trajectory_results[label] = {
            "label": label,
            "trial": trial,
            "flip_active": flip_active,
            "fm_delta_loss_check": float(fm_loss_check - baseline),
            "pred_xyz": pred_xyz,
            "pred_rot": pred_rot,
            "cot": extra["cot"][0].tolist() if "cot" in extra else None,
            "metrics": trajectory_metrics(pred_xyz),
            "delta_vs_clean": trajectory_delta_metrics(pred_xyz, clean_pred_xyz),
        }
    except Exception as exc:
        trajectory_results[label] = {
            "label": label,
            "trial": trial,
            "error": repr(exc),
        }
    finally:
        bfa.restore_one(mod.weight.data, flat_idx, orig)

torch.save(trajectory_results, TRAJ_OUT_PATH)
metrics_summary = {
    key: {
        "trial": value.get("trial"),
        "flip_active": value.get("flip_active"),
        "fm_delta_loss_check": value.get("fm_delta_loss_check"),
        "metrics": value.get("metrics"),
        "delta_vs_clean": value.get("delta_vs_clean"),
        "error": value.get("error"),
    }
    for key, value in trajectory_results.items()
}
print("\nFull-rollout deltas vs clean:")
for key, summary in metrics_summary.items():
    if key == "clean":
        continue
    print(key, summary["delta_vs_clean"], "fm_delta_loss_check=", summary["fm_delta_loss_check"])
TRAJ_METRICS_PATH.write_text(json.dumps(metrics_summary, indent=2))
print(f"saved full-rollout trajectories to {TRAJ_OUT_PATH}")
print(f"saved metrics summary to {TRAJ_METRICS_PATH}")

In [ ]:
# Plot the saved full-rollout trajectories, matching inference_cam_num.ipynb style.
trajectory_results = torch.load(TRAJ_OUT_PATH, map_location="cpu")

print("Pairwise max |xy delta| between saved rollout groups:")
pred_items = [(k, v["pred_xyz"]) for k, v in trajectory_results.items() if "pred_xyz" in v]
for i, (name_a, pred_a) in enumerate(pred_items):
    for name_b, pred_b in pred_items[i + 1:]:
        a = pred_a[0, 0, :, :, :2].numpy()
        b = pred_b[0, 0, :, :, :2].numpy()
        valid = np.isfinite(a).all(axis=(1, 2)) & np.isfinite(b).all(axis=(1, 2))
        if valid.any():
            max_delta = float(np.abs(a[valid] - b[valid]).max())
            print(f"{name_a} vs {name_b}: {max_delta:.6g} m")
        else:
            print(f"{name_a} vs {name_b}: no comparable finite samples")

gt_xy = data["ego_future_xyz"].cpu()[0, 0, :, :2].T.numpy()
colors = ["tab:blue", "tab:orange", "tab:green", "tab:purple", "tab:brown", "tab:pink"]

plt.figure(figsize=(7, 7))
for idx, (label, res) in enumerate(trajectory_results.items()):
    if "pred_xyz" not in res:
        print(f"Skipping {label}: {res.get('error', 'no pred_xyz saved')}")
        continue
    color = colors[idx % len(colors)]
    pred_xys = res["pred_xyz"][0, 0, :, :, :2].numpy()
    valid = np.isfinite(pred_xys).all(axis=(1, 2))
    if not valid.any():
        print(f"Skipping {label}: all predicted trajectories are non-finite")
        continue
    short_label = label if label == "clean" else f"BFA {idx}"
    for sample_idx, pred_xy in enumerate(pred_xys[valid]):
        plt.plot(
            pred_xy.T[0],
            pred_xy.T[1],
            "o-",
            color=color,
            markersize=3,
            alpha=0.25,
            label=short_label if sample_idx == 0 else None,
        )

plt.plot(gt_xy[0], gt_xy[1], "r-", linewidth=2, label="Ground Truth")
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title("Trajectory Comparison After Selected BFA Flips")
plt.legend(loc="best", fontsize=8)
plt.axis("equal")
plt.tight_layout()
plt.savefig("bfa_trajectory_comparison.png", dpi=120)
plt.show()